In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from conus_biomass import dir_info

In [ ]:
cmip_tseries = xr.open_dataset("figure_data/CMIP_tseries.nc")

In [ ]:
df_all_westwide = pd.read_csv("figure_data/OurStudy_western_stocks.csv")
western_stocks = pd.read_csv("figure_data/USFS_western_stocks.csv")
western_stocks = western_stocks.rename(columns={"Unnamed: 0": "year", "0": "live_biomass_MMT"})

In [ ]:
biomass_mean_west = df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).median()
biomass_min_west = (
    df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).quantile(0.025)
)
biomass_max_west = (
    df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).quantile(0.975)
)

biomass_mean_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).median()
)
biomass_min_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).quantile(0.025)
)
biomass_max_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).quantile(0.975)
)

In [ ]:
tseries_df = xr.open_dataset("figure_data/Liu_tseries.nc")
tseries_Liu = tseries_df["Liu_biomass"]

In [ ]:
trends_df = pd.read_csv("figure_data/Li_trends.csv")
trend1_westwide = trends_df["trend"].values[0]
startyr1 = trends_df["year_start"].values[0]
endyr1 = trends_df["year_end"].values[0]
trend2_westwide = trends_df["trend"].values[1]
startyr2 = trends_df["year_start"].values[1]
endyr2 = trends_df["year_end"].values[1]

In [ ]:
cmip_times = cmip_tseries["change_since_2005"].year.values

p10 = cmip_tseries["change_since_2005"].quantile(0.1, dim="ensemble_member")
p50 = cmip_tseries["change_since_2005"].mean(dim="ensemble_member")
p90 = cmip_tseries["change_since_2005"].quantile(0.9, dim="ensemble_member")

In [ ]:
import figure_settings
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

figure_settings.apply_style()

lwidth = 5
plt.figure(figsize=(6, 5))

# Stack all CMIP6 anomaly time series and compute percentile envelope
plt.fill_between(cmip_times, p10, p90, color="#bc85d9", alpha=0.1, linewidth=0)
plt.plot(cmip_times, p50, color="#bc85d9", linewidth=lwidth)
plt.plot(
    biomass_mean_west.index,
    (np.array(biomass_mean_west)) - (np.array(biomass_mean_west))[0],
    ".-",
    linewidth=5,
    color="#85a2f7",
    label="This Study",
)
tseries = tseries_Liu
plt.plot(
    np.arange(2005, 2021),
    tseries - tseries[0],
    linewidth=lwidth,
    label="Liu et al. 2025",
    color="#e587b6",
)
plt.plot(
    western_stocks["year"],
    western_stocks["live_biomass_MMT"] - western_stocks["live_biomass_MMT"][15],
    "-",
    linewidth=5,
    color="gray",
    label="USFS/EPA Estimate",
)
plt.plot(
    [2010, 2016],
    [trend1_westwide * 5, trend1_westwide * 11],
    "--.",
    color="#64b9c4",
    linewidth=lwidth,
    markersize=20,
)
plt.plot(
    [2016, 2022],
    [trend1_westwide * 11, trend1_westwide * 11 + trend2_westwide * 6],
    "--.",
    color="#64b9c4",
    linewidth=lwidth,
    markersize=20,
    label="Li et al. 2025",
)
plt.xlim([2004.5, 2023.5])
plt.axhline(y=0, linestyle=":", color="gray")
cmip_handle = (
    Patch(facecolor="#bc85d9", alpha=0.3, linewidth=0),
    Line2D([0], [0], color="#bc85d9", linewidth=lwidth),
)
handles, labels = plt.gca().get_legend_handles_labels()
handles = [cmip_handle] + handles
labels = ["CMIP6"] + labels
plt.legend(handles, labels, framealpha=False)
plt.ylabel("Cumulative change in western \n live forest biomass (MMT C)")

ax = plt.gca()
ax.set_xticks(np.arange(2005, 2024, 5))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}"))
ax.tick_params(which="both", direction="in", top=True, right=True)

plt.tight_layout()
plt.savefig(dir_info.dir_figures + "Figure5.jpg")